# analysis_manifest_backfill_v1

目标：验证旧 fixed-sidecar research card 可以通过 `src.analysis.backfill_analysis_manifest` 补出 `analysis_manifest.json`，然后被 dashboard manifest-first 路径读取。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.analysis import backfill_analysis_manifest
from src.dashboard import load_model_matrix_dashboard

CARD = Path('research/analysis_manifest_backfill_v1')
LEGACY_CARD = CARD / 'legacy_card'
CARD.mkdir(parents=True, exist_ok=True)
LEGACY_CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'legacy_card': str(LEGACY_CARD),
}


{'card': 'research/analysis_manifest_backfill_v1',
 'legacy_card': 'research/analysis_manifest_backfill_v1/legacy_card'}

## 构造旧 fixed-sidecar card

这个 card 只有固定文件名 sidecar，没有 `analysis_manifest.json`。同时放入 `predictions.csv` 和 `daily_backtest.csv`，用于验证 backfill 不把私有/原始产物暴露给 dashboard。

In [2]:
summary = {
    'schema_version': 'analysis.report.v1',
    'status': 'legacy_fixed_sidecars_completed',
    'start': '2026-01-01',
    'end': '2026-06-09',
    'model_count': 2,
}
(LEGACY_CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame([
    {'rank': 1, 'model': 'ridge_mse', 'return_pct': 3.2, 'relative_return_pct': 1.1, 'stability_score': 2.4},
    {'rank': 2, 'model': 'gcn_mse', 'return_pct': -1.3, 'relative_return_pct': -2.0, 'stability_score': -1.2},
]).to_csv(LEGACY_CARD / 'model_comparison.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 2.5},
    {'model': 'gcn_mse', 'regime': 'stress', 'return_pct': -4.0},
]).to_csv(LEGACY_CARD / 'regime_metrics.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse', 'window': 'test', 'return_pct': 3.2},
    {'model': 'gcn_mse', 'window': 'test', 'return_pct': -1.3},
]).to_csv(LEGACY_CARD / 'window_metrics.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse', 'return_kind': 'strategy_return', 'annualized_return': 0.31},
]).to_csv(LEGACY_CARD / 'qlib_risk_metrics.csv', index=False)
pd.DataFrame([
    {'model': 'gcn_mse', 'segment_type': 'regime', 'segment': 'stress', 'regime': 'stress', 'diagnosis': '弱于基准：压力期收益落后。'},
]).to_csv(LEGACY_CARD / 'model_effect_deltas.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse', 'paper_fidelity': 'native_library'},
    {'model': 'gcn_mse', 'paper_fidelity': 'proxy_feature_adapter'},
]).to_csv(LEGACY_CARD / 'model_capabilities.csv', index=False)
pd.DataFrame([{'model': 'ridge_mse', 'prediction': 0.1}]).to_csv(LEGACY_CARD / 'predictions.csv', index=False)
pd.DataFrame([{'model': 'ridge_mse', 'nav': 1.0}]).to_csv(LEGACY_CARD / 'daily_backtest.csv', index=False)
sorted(path.name for path in LEGACY_CARD.iterdir())


['analysis_summary.json',
 'daily_backtest.csv',
 'model_capabilities.csv',
 'model_comparison.csv',
 'model_effect_deltas.csv',
 'predictions.csv',
 'qlib_risk_metrics.csv',
 'regime_metrics.csv',
 'window_metrics.csv']

## Backfill analysis_manifest.json

使用稳定 `src/analysis` 接口生成 manifest。这个 manifest 只登记 dashboard/notebook/promotion 可公开消费的 sidecar。

In [3]:
manifest = backfill_analysis_manifest(
    LEGACY_CARD,
    dataset_name='legacy_dashboard_card',
    universe=('SOXL', 'SOXX'),
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(LEGACY_CARD),),
)
assert (LEGACY_CARD / 'analysis_manifest.json').exists()
assert manifest['metadata']['generated_from'] == 'legacy_sidecars'
assert len(manifest['manifest_hash']) == 64
assert 'predictions' not in manifest['paths']
assert 'daily_backtest' not in manifest['paths']
manifest


{'artifact_id': 'legacy_card',
 'artifact_type': 'analysis_report',
 'dataset': {'calendar_path': 'data/qlib/soxl/calendars/day.txt',
  'data_paths': ['research/analysis_manifest_backfill_v1/legacy_card'],
  'end': '2026-06-09',
  'metadata': {'source': 'analysis_manifest_backfill'},
  'name': 'legacy_dashboard_card',
  'schema_version': 'dataset.v1',
  'start': '2026-01-01',
  'universe': ['SOXL', 'SOXX']},
 'metadata': {'analysis_report_schema_version': 'analysis.report.v1',
  'generated_from': 'legacy_sidecars',
  'table_counts': {'model_capabilities': 2,
   'model_comparison': 2,
   'model_effect_deltas': 1,
   'qlib_risk_metrics': 1,
   'regime_metrics': 2,
   'window_metrics': 2}},
 'paths': {'model_capabilities': 'model_capabilities.csv',
  'model_comparison': 'model_comparison.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'regime_metrics': 'regime_metrics.csv',
  'summary': 'analysis_summary.json',
  'window_metrics'

## Dashboard 读取验证

补 manifest 后，dashboard 应该走 `analysis_manifest` 来源，而不是 `fixed_sidecars` fallback。

In [4]:
payload = load_model_matrix_dashboard(LEGACY_CARD, top_n=2)
assert payload['manifest_source']['kind'] == 'analysis_manifest'
assert payload['manifest_source']['manifest_hash'] == manifest['manifest_hash']
assert payload['top_models'][0]['model'] == 'ridge_mse'
assert 'predictions' not in payload['artifacts']
assert 'daily_backtest' not in payload['artifacts']
{
    'artifact_id': payload['artifact_id'],
    'manifest_source': payload['manifest_source'],
    'artifact_keys': sorted(payload['artifacts']),
    'top_models': [row['model'] for row in payload['top_models']],
}


{'artifact_id': 'legacy_card',
 'manifest_source': {'kind': 'analysis_manifest',
  'path': 'research/analysis_manifest_backfill_v1/legacy_card/analysis_manifest.json',
  'schema_version': 'artifact.v1',
  'manifest_hash': '5a966111a4061dc1ce76c3a5148927c79ef6c070b61069c51770a132e2d898c8',
  'producer': 'src.analysis.manifest_backfill'},
 'artifact_keys': ['analysis_manifest',
  'model_capabilities',
  'model_comparison',
  'model_effect_deltas',
  'qlib_risk_metrics',
  'regime_metrics',
  'summary',
  'window_metrics'],
 'top_models': ['ridge_mse', 'gcn_mse']}

## 写出本实验摘要

根目录摘要记录 backfill 的 manifest hash 和 dashboard 读取结果。

In [5]:
run_summary = {
    'schema_version': 'research.analysis_manifest_backfill_v1',
    'status': 'completed',
    'legacy_card': str(LEGACY_CARD),
    'manifest_path': str(LEGACY_CARD / 'analysis_manifest.json'),
    'manifest_hash': manifest['manifest_hash'],
    'manifest_paths': manifest['paths'],
    'dashboard_manifest_kind': payload['manifest_source']['kind'],
    'dashboard_artifact_id': payload['artifact_id'],
    'private_sidecars_hidden': 'predictions' not in payload['artifacts'] and 'daily_backtest' not in payload['artifacts'],
    'conclusion': '旧 fixed-sidecar card 可以补出 analysis_manifest.json，并由 dashboard manifest-first 路径读取。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary


{'schema_version': 'research.analysis_manifest_backfill_v1',
 'status': 'completed',
 'legacy_card': 'research/analysis_manifest_backfill_v1/legacy_card',
 'manifest_path': 'research/analysis_manifest_backfill_v1/legacy_card/analysis_manifest.json',
 'manifest_hash': '5a966111a4061dc1ce76c3a5148927c79ef6c070b61069c51770a132e2d898c8',
 'manifest_paths': {'model_capabilities': 'model_capabilities.csv',
  'model_comparison': 'model_comparison.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'regime_metrics': 'regime_metrics.csv',
  'summary': 'analysis_summary.json',
  'window_metrics': 'window_metrics.csv'},
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_artifact_id': 'legacy_card',
 'private_sidecars_hidden': True,
 'conclusion': '旧 fixed-sidecar card 可以补出 analysis_manifest.json，并由 dashboard manifest-first 路径读取。'}

## 中文结论

- 做了什么：新增并验证旧 analysis sidecar card 的 `analysis_manifest.json` backfill 流程。
- 核心指标：生成的 manifest 带 64 位 hash，登记 summary、model_comparison、regime/window、qlib_risk、model_effect_deltas、model_capabilities 等公开 sidecar。
- regime 差异：本轮 legacy card 的 bull/stress regime 表通过 manifest 被 dashboard 读取；压力期诊断仍在 `model_effect_deltas`。
- 是否晋升：这是框架迁移能力，不是策略晋升；它让旧研究 card 逐步进入统一 manifest/dashboard 体系。
- 风险：backfill 不推断缺失表，也不重算指标；如果旧 card 缺 sidecar，需要先补实验产物或重跑 notebook。
- 下一步：可以提供批量扫描 research/ 的脚本，但默认仍应人工选择 card，避免改动大量历史产物。